# Active Share Analysis: Tuleva Täiendav Kogumisfond vs MSCI ACWI

**11 March 2026**

Look-through analysis of Tuleva Täiendav Kogumisfond (Additional Savings Fund) model portfolio holdings vs the MSCI ACWI benchmark.
Active share measures how much the fund's portfolio differs from the index.

**Data sources:**
- TKF model portfolio — [published PDF](https://tuleva.ee/wp-content/uploads/2026/01/Tuleva-Taiendav-Kogumisfond-mudelportfell-avalikustamiseks.pdf)
- ETF holdings (look-through) — downloaded from each ETF provider's website
- ACWI benchmark — iShares MSCI ACWI UCITS ETF (SSAC) via iShares.com

In [1]:
import sys
import os
import io
import re
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import HTML, display

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from generate_charts import setup_plot_style, TULEVA_BLUE, TULEVA_NAVY, TULEVA_MID_BLUE

setup_plot_style()

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Brand colors
COLOR_OVERWEIGHT = TULEVA_NAVY
COLOR_UNDERWEIGHT = '#FF4800'
COLOR_NEUTRAL = '#B0B0B0'

# iShares config (for SSAC benchmark + iShares Developed World fund)
ISHARES_AJAX = '1506575576011.ajax'
ISHARES_BASE = 'https://www.ishares.com/uk/individual/en/products'
ISHARES_UA = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'

ISHARES_PRODUCTS = {
    'SAWD': {'id': 305419, 'slug': 'ishares-msci-world-esg-screened-ucits-etf-usd-acc-fund'},
    'SSAC': {'id': 251850, 'slug': 'ishares-msci-acwi-ucits-etf'},
    # SSAC sub-ETFs for look-through
    'NDIA': {'id': 297617, 'slug': 'ishares-msci-india-ucits-etf'},
    '4BRZ': {'id': 304304, 'slug': 'ishares-msci-brazil-ucits-etf-usd-hedged'},
    'CNYA': {'id': 273192, 'slug': 'ishares-msci-china-a-ucits-etf'},
    'IKSA': {'id': 279996, 'slug': 'ishares-msci-saudi-arabia-capped-imi-ucits-etf'},
}

SSAC_SUB_ETFS = {'NDIA', '4BRZ', 'CNYA', 'IKSA'}

# Local holdings files
HOLDINGS_DIR = Path.cwd() / 'data' / 'tkf_holdings'

# EM countries (for separating DM/EM in ACWI)
EM_COUNTRIES = {
    'China', 'Taiwan', 'India', 'Korea (South)', 'Brazil', 'South Africa',
    'Mexico', 'Saudi Arabia', 'Thailand', 'Indonesia', 'Malaysia', 'Philippines',
    'Turkey', 'Poland', 'Chile', 'Qatar', 'United Arab Emirates', 'Kuwait',
    'Colombia', 'Peru', 'Czech Republic', 'Egypt', 'Greece', 'Hungary',
}


def fetch_ishares_holdings(ticker):
    """Download full holdings CSV from iShares website."""
    product = ISHARES_PRODUCTS[ticker]
    url = (f'{ISHARES_BASE}/{product["id"]}/{product["slug"]}'
           f'/{ISHARES_AJAX}?fileType=csv&fileName={ticker}_holdings&dataType=fund')
    resp = requests.get(url, headers={'User-Agent': ISHARES_UA})
    resp.raise_for_status()
    lines = resp.text.splitlines()
    date_line = lines[0] if lines else ''
    holdings_date = date_line.split(',')[1].strip().strip('"') if ',' in date_line else None
    csv_start = next(i for i, l in enumerate(lines) if l.startswith('Ticker,'))
    df = pd.read_csv(io.StringIO('\n'.join(lines[csv_start:])), thousands=',')
    df['Weight (%)'] = pd.to_numeric(df['Weight (%)'], errors='coerce')
    return df, holdings_date


def is_equity(row):
    return row.get('Asset Class', '') == 'Equity'


def normalize_em_name(name):
    """Aggressively normalize stock names for cross-provider matching."""
    n = str(name).upper().strip()
    n = re.sub(r'\s*\([A-Z]+\)\s*', ' ', n)
    n = re.sub(r'\s+(KRW|TWD|INR|CNY|HKD|ZAR|SAR|BRL|MXN|THB|IDR|MYR|PHP|PLN|CLP|QAR|AED|KWD|COP|PEN|CZK|EGP|HUF|TRY|USD|NPV)\s*[\d.]*', '', n)
    n = re.sub(r'\s*-?(?:PFD|PREF|NON VOTING PRE|CLASS [A-Z]|CL [A-Z])\s*$', '', n)
    n = re.sub(r'\s*-[A-Z]+\s*$', '', n)
    for _ in range(3):
        n = re.sub(r'\s*(CORPORATION|CORP|LIMITED|LTD|INC|PLC|CO|SA|AG|NV|SE|SPA|AB|ASA|OYJ|TBK|BHD|PCL|BERHAD|SH|ADS)\s*$', '', n)
    n = n.replace(' & ', ' AND ')
    n = re.sub(r'\s+', ' ', n).strip()
    return n


def scrollable_table(df, title='', max_height=400, fmt=None):
    """Render a DataFrame as a scrollable HTML table."""
    styled = df.style
    if fmt:
        styled = styled.format(fmt)
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [('position', 'sticky'), ('top', '0'),
                                      ('background', '#002F63'), ('color', 'white'),
                                      ('padding', '6px 10px'), ('font-size', '0.85em')]},
        {'selector': 'td', 'props': [('padding', '4px 10px'), ('font-size', '0.85em'),
                                      ('border-bottom', '1px solid #eee')]},
        {'selector': 'tr:hover td', 'props': [('background', '#f5f5f5')]},
    ])
    html = styled.to_html()
    title_html = f'<h4 style="margin: 15px 0 5px 0;">{title}</h4>' if title else ''
    footer = f'<p style="color: #666; font-size: 0.85em; margin: 3px 0 15px 0;">{len(df)} rows</p>'
    return HTML(f'{title_html}<div style="max-height: {max_height}px; overflow-y: auto; '
                f'border: 1px solid #ddd; margin: 5px 0;">{html}</div>{footer}')


def metric_box(label, value, note=''):
    """Render a key metric as styled HTML."""
    note_html = f'<p style="color:#666; font-size:0.85em; margin:3px 0 0 0;">{note}</p>' if note else ''
    return HTML(f'<div style="background:#f8f9fa; border-left:4px solid #002F63; padding:12px 16px; '
                f'margin:10px 0; border-radius:0 4px 4px 0;">'
                f'<span style="font-size:0.9em; color:#666;">{label}</span><br>'
                f'<span style="font-size:1.8em; font-weight:bold; color:#002F63;">{value}</span>'
                f'{note_html}</div>')


def diff_table(df, weight_col, bench_col, name_col='name', title='', n=20):
    """Render a scrollable table of weight differences."""
    t = df.sort_values('abs_diff', ascending=False).head(n).copy()
    t['Direction'] = t['diff'].apply(lambda x: 'OW' if x > 0 else 'UW')
    out = t[[name_col, weight_col, bench_col, 'diff', 'Direction']].copy()
    out.columns = ['Stock', 'Fund %', 'Bench %', 'Diff', '']
    out.index = range(1, len(out) + 1)
    return scrollable_table(out, title=title,
                           fmt={'Fund %': '{:.2%}', 'Bench %': '{:.2%}', 'Diff': '{:+.2%}'})


# Global ISIN → Ticker|Location mapping
isin_to_ticker_loc = {}

def register_isin_mapping(isin, ticker, location):
    if pd.notna(isin) and pd.notna(ticker) and pd.notna(location):
        isin_to_ticker_loc[str(isin)] = f"{ticker}|{location}"


def stock_id_ticker_loc(row):
    ticker = row.get('ticker') or row.get('Ticker') or '?'
    country = row.get('country') or row.get('Location') or '?'
    return f"{ticker}|{country}"


def stock_id_isin(row):
    isin = row.get('isin') or row.get('ISIN')
    if pd.notna(isin) and len(str(isin)) == 12 and str(isin)[:2].isalpha():
        return str(isin)
    return None


print('Setup complete')

Setup complete


In [2]:
# TKF model portfolio (from published PDF, Dec 2025)
model_portfolio = pd.DataFrame([
    {'name': 'Invesco MSCI EM Universal Screened UCITS ETF Acc',       'isin': 'IE00BMDBMY19', 'weight': 10.00, 'source': 'companiesmarketcap'},
    {'name': 'iShares Developed World Screened Index Fund',            'isin': 'IE00BFG1TM61', 'weight': 19.00, 'source': 'ishares_SAWD'},
    {'name': 'Xtrackers MSCI USA Screened UCITS ETF',                  'isin': 'IE00BJZ2DC62', 'weight': 17.40, 'source': 'xtrackers_usa'},
    {'name': 'Xtrackers MSCI Canada Screened UCITS ETF',               'isin': 'LU0476289540', 'weight':  1.60, 'source': 'xtrackers_canada'},
    {'name': 'ICAV Amundi MSCI USA Screened UCITS ETF',                'isin': 'IE000F60HVH9', 'weight': 18.50, 'source': 'amundi_usa'},
    {'name': 'Vanguard ESG North America All Cap UCITS ETF',           'isin': 'IE000O58J820', 'weight': 15.40, 'source': 'vanguard_na'},
    {'name': 'BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF',         'isin': 'LU1291099718', 'weight': 12.50, 'source': 'bnp_europe'},
    {'name': 'BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF','isin': 'LU1291106356', 'weight':  1.90, 'source': 'bnp_pacific'},
    {'name': 'BNP Paribas Easy MSCI Japan Min TE UCITS ETF',          'isin': 'LU1291102447', 'weight':  3.70, 'source': 'bnp_japan'},
])

mp_display = model_portfolio[['name', 'isin', 'weight']].copy()
mp_display.index = range(1, len(mp_display) + 1)
display(scrollable_table(mp_display, title='TKF Model Portfolio', max_height=350,
                         fmt={'weight': '{:.1f}%'}))

,name,isin,weight
1,Invesco MSCI EM Universal Screened UCITS ETF Acc,IE00BMDBMY19,10.0%
2,iShares Developed World Screened Index Fund,IE00BFG1TM61,19.0%
3,Xtrackers MSCI USA Screened UCITS ETF,IE00BJZ2DC62,17.4%
4,Xtrackers MSCI Canada Screened UCITS ETF,LU0476289540,1.6%
5,ICAV Amundi MSCI USA Screened UCITS ETF,IE000F60HVH9,18.5%
6,Vanguard ESG North America All Cap UCITS ETF,IE000O58J820,15.4%
7,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,LU1291099718,12.5%
8,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,LU1291106356,1.9%
9,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,LU1291102447,3.7%


In [3]:
# Load ETF holdings from all sources
# Each loader returns a DataFrame with columns: name, isin, weight (fraction 0-1), sector, country
# iShares and Vanguard lack ISINs — they use ticker-based IDs instead

all_etf_holdings = {}  # source_key → DataFrame
holdings_dates = {}


def load_xtrackers(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, skiprows=3,
                       names=['#', 'Name', 'ISIN', 'Country', 'Currency', 'Exchange',
                              'Type', 'Rating', 'Primary_Listing', 'Sector', 'Weight'])
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    return df[['Name', 'ISIN', 'Country', 'Sector', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_amundi(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, engine='calamine', header=None, skiprows=20)
    df.columns = ['_skip', 'ISIN', 'Name', 'Asset_Class', 'Currency', 'Weight', 'Sector', 'Country'] + \
                 [f'_c{i}' for i in range(max(0, len(df.columns) - 8))]
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    return df[['Name', 'ISIN', 'Country', 'Sector', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_vanguard(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, skiprows=6)
    df.columns = ['Ticker', 'Name', 'Weight', 'Sector', 'Region'] + list(df.columns[5:])
    df = df.dropna(subset=['Name'])
    df['Weight'] = df['Weight'].astype(str).str.rstrip('%').astype(float) / 100
    df['Country'] = df['Region'].map({'US': 'United States', 'CA': 'Canada'})
    return df[['Ticker', 'Name', 'Country', 'Sector', 'Weight']].rename(
        columns={'Ticker': 'ticker', 'Name': 'name', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_bnp(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, header=None, engine='calamine', skiprows=22)
    df.columns = ['_skip', 'Asset_Class', 'Name', 'ISIN', 'Weight'] + \
                 [f'_c{i}' for i in range(max(0, len(df.columns) - 5))]
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    df = df[df['Asset_Class'] == 'Equity']
    return df[['Name', 'ISIN', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Weight': 'weight'})


def load_invesco_from_web():
    """Fetch Invesco MSCI EM Universal Screened holdings from companiesmarketcap.com."""
    url = 'https://companiesmarketcap.com/gbp/invesco-msci-emerging-markets-universal-screened-ucits-etf/holdings/'
    r = requests.get(url, headers={'User-Agent': ISHARES_UA})
    soup = BeautifulSoup(r.text, 'html.parser')
    table = soup.find('table')
    rows = []
    for tr in table.find_all('tr')[1:]:
        cells = tr.find_all('td')
        if len(cells) >= 3:
            weight_text = cells[0].get_text(strip=True).rstrip('%')
            name = cells[1].get_text(strip=True)
            isin = cells[2].get_text(strip=True)
            try:
                weight = float(weight_text) / 100
            except ValueError:
                continue
            rows.append({'name': name, 'isin': isin, 'weight': weight})
    return pd.DataFrame(rows)


# --- Load all holdings ---

# 1. Xtrackers USA
print('Loading Xtrackers USA...', end=' ')
all_etf_holdings['xtrackers_usa'] = load_xtrackers('Constituent_IE00BJZ2DC62.xlsx')
holdings_dates['xtrackers_usa'] = 'Mar 2026'
print(f'{len(all_etf_holdings["xtrackers_usa"])} holdings')

# 2. Xtrackers Canada
print('Loading Xtrackers Canada...', end=' ')
all_etf_holdings['xtrackers_canada'] = load_xtrackers('Constituent_LU0476289540.xlsx')
holdings_dates['xtrackers_canada'] = 'Mar 2026'
print(f'{len(all_etf_holdings["xtrackers_canada"])} holdings')

# 3. Amundi USA
print('Loading Amundi USA...', end=' ')
all_etf_holdings['amundi_usa'] = load_amundi(
    'Fund Holdings_Amundi MSCI USA Screened UCITS ETF Acc_IE000F60HVH9_09_03_2026.xlsx')
holdings_dates['amundi_usa'] = '9 Mar 2026'
print(f'{len(all_etf_holdings["amundi_usa"])} holdings')

# 4. Vanguard NA (stale: 31 Jan 2026 — estimated effect on active share < 1.5%)
print('Loading Vanguard NA...', end=' ')
all_etf_holdings['vanguard_na'] = load_vanguard(
    'Holdings details - Vanguard ESG North America All Cap UCITS ETF (USD) Accumulating - 3_11_2026.xlsx')
holdings_dates['vanguard_na'] = '31 Jan 2026 (stale — estimated effect on active share < 1.5%)'
print(f'{len(all_etf_holdings["vanguard_na"])} holdings')

# 5. BNP Europe
print('Loading BNP Europe...', end=' ')
all_etf_holdings['bnp_europe'] = load_bnp('Fund_Holdings_INV_LU1291099718_en.xlsx')
holdings_dates['bnp_europe'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_europe"])} holdings')

# 6. BNP Pacific ex Japan
print('Loading BNP Pacific ex Japan...', end=' ')
all_etf_holdings['bnp_pacific'] = load_bnp('Fund_Holdings_INV_LU1291106356_en.xlsx')
holdings_dates['bnp_pacific'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_pacific"])} holdings')

# 7. BNP Japan
print('Loading BNP Japan...', end=' ')
all_etf_holdings['bnp_japan'] = load_bnp('Fund_Holdings_INV_LU1291102447_en.xlsx')
holdings_dates['bnp_japan'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_japan"])} holdings')

# 8. iShares Developed World Screened (fetched live — no ISIN column, uses Ticker|Location)
print('Fetching iShares SAWD...', end=' ')
sawd_raw, sawd_date = fetch_ishares_holdings('SAWD')
sawd_eq = sawd_raw[sawd_raw.apply(is_equity, axis=1)].copy()
sawd_eq['weight'] = sawd_eq['Weight (%)'] / 100
sawd_eq = sawd_eq.rename(columns={'Name': 'name', 'Location': 'country', 'Sector': 'sector', 'Ticker': 'ticker'})
all_etf_holdings['ishares_SAWD'] = sawd_eq[['ticker', 'name', 'country', 'sector', 'weight']]
holdings_dates['ishares_SAWD'] = sawd_date
print(f'{len(sawd_eq)} equities (as of {sawd_date})')
time.sleep(0.5)

# 9. Invesco EM (fetched from companiesmarketcap)
print('Fetching Invesco EM from companiesmarketcap.com...', end=' ')
all_etf_holdings['companiesmarketcap'] = load_invesco_from_web()
holdings_dates['companiesmarketcap'] = 'Mar 2026'
print(f'{len(all_etf_holdings["companiesmarketcap"])} holdings')

# Summary
print(f'\n{"Source":<25s} {"Holdings":>8s}  {"Weight sum":>10s}  Date')
print('-' * 70)
for key, df in all_etf_holdings.items():
    print(f'{key:<25s} {len(df):>8d}  {df["weight"].sum():>10.4f}  {holdings_dates[key]}')

Loading Xtrackers USA... 486 holdings
Loading Xtrackers Canada... 76 holdings
Loading Amundi USA... 483 holdings
Loading Vanguard NA... 

1415 holdings
Loading BNP Europe... 362 holdings
Loading BNP Pacific ex Japan... 89 holdings
Loading BNP Japan... 158 holdings
Fetching iShares SAWD... 

1201 equities (as of 09/Mar/2026)


Fetching Invesco EM from companiesmarketcap.com... 

472 holdings

Source                    Holdings  Weight sum  Date
----------------------------------------------------------------------
xtrackers_usa                  486      1.0000  Mar 2026
xtrackers_canada                76      1.0000  Mar 2026
amundi_usa                     483      1.0005  9 Mar 2026
vanguard_na                   1415      0.9933  31 Jan 2026 (stale — estimated effect on active share < 1.5%)
bnp_europe                     362      0.9954  6 Mar 2026
bnp_pacific                     89      0.9921  6 Mar 2026
bnp_japan                      158      0.9583  6 Mar 2026
ishares_SAWD                  1201      0.9960  09/Mar/2026
companiesmarketcap             472      0.9298  Mar 2026


In [4]:
# Build look-through TKF portfolio
# Use OpenFIGI ISIN→Ticker|Location mapping for cross-referencing with iShares ACWI data
import json

with open(HOLDINGS_DIR / 'isin_ticker_map.json') as f:
    isin_to_sid = json.load(f)

print(f'Loaded ISIN→Ticker|Location mapping: {len(isin_to_sid)} ISINs')

# Build look-through portfolio
all_stocks = []

for _, mp_row in model_portfolio.iterrows():
    source = mp_row['source']
    fund_weight = mp_row['weight'] / 100.0
    df_h = all_etf_holdings[source]

    for _, h in df_h.iterrows():
        # Determine stock_id: prefer Ticker|Location, fall back to ISIN lookup
        if pd.notna(h.get('ticker')) and pd.notna(h.get('country')):
            sid = f"{h['ticker']}|{h['country']}"
        elif pd.notna(h.get('isin')) and str(h.get('isin')) in isin_to_sid:
            sid = isin_to_sid[str(h['isin'])]
        elif pd.notna(h.get('isin')):
            sid = str(h['isin'])  # fallback: bare ISIN
        else:
            sid = f"UNKNOWN|{h['name']}"

        all_stocks.append({
            'stock_id': sid,
            'name': h['name'],
            'isin': h.get('isin', ''),
            'weight': fund_weight * h['weight'] * 100,
            'sector': h.get('sector', ''),
            'country': h.get('country', ''),
            'source_etf': source,
        })

    print(f'{mp_row["name"][:50]:50s}  {mp_row["weight"]:5.1f}% × {len(df_h):>5d} stocks')

tkf_raw = pd.DataFrame(all_stocks)

# Aggregate by stock_id (same stock from multiple ETFs)
tkf_portfolio = (tkf_raw
    .groupby('stock_id')
    .agg({'name': 'first', 'isin': 'first', 'weight': 'sum', 'sector': 'first', 'country': 'first'})
    .reset_index()
)

# Stats on matching
ticker_loc_ids = tkf_portfolio[tkf_portfolio['stock_id'].str.contains('|', regex=False, na=False)]
isin_only = tkf_portfolio[~tkf_portfolio['stock_id'].str.contains('|', regex=False, na=False)]
print(f'\nStock ID types: {len(ticker_loc_ids)} Ticker|Location, {len(isin_only)} unmatched ({isin_only["weight"].sum():.2f}% weight)')

# Normalize to 100%
total_weight = tkf_portfolio['weight'].sum()
tkf_portfolio['weight'] = tkf_portfolio['weight'] * (100 / total_weight)
tkf_portfolio = tkf_portfolio.sort_values('weight', ascending=False).reset_index(drop=True)

print(f'TKF look-through: {len(tkf_portfolio):,} unique stocks (normalized from {total_weight:.2f}%)')
print(f'\nTop 20 holdings:')
display(tkf_portfolio.head(20)[['stock_id', 'name', 'weight', 'sector', 'country']]
        .style.format({'weight': '{:.4f}%'}))

Loaded ISIN→Ticker|Location mapping: 1642 ISINs
Invesco MSCI EM Universal Screened UCITS ETF Acc     10.0% ×   472 stocks
iShares Developed World Screened Index Fund          19.0% ×  1201 stocks
Xtrackers MSCI USA Screened UCITS ETF                17.4% ×   486 stocks
Xtrackers MSCI Canada Screened UCITS ETF              1.6% ×    76 stocks
ICAV Amundi MSCI USA Screened UCITS ETF              18.5% ×   483 stocks
Vanguard ESG North America All Cap UCITS ETF         15.4% ×  1415 stocks
BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF        12.5% ×   362 stocks
BNP Paribas Easy MSCI Pacific ex Japan Min TE UCIT    1.9% ×    89 stocks
BNP Paribas Easy MSCI Japan Min TE UCITS ETF          3.7% ×   158 stocks

Stock ID types: 2758 Ticker|Location, 6 unmatched (-0.02% weight)
TKF look-through: 2,764 unique stocks (normalized from 98.90%)

Top 20 holdings:


,stock_id,name,weight,sector,country
0,NVDA|United States,NVIDIA CORP,5.3303%,Information Technology,United States
1,AAPL|United States,APPLE INC,4.5698%,Information Technology,United States
2,MSFT|United States,MICROSOFT CORP,3.5493%,Information Technology,United States
3,AMZN|United States,AMAZON COM INC,2.5306%,Consumer Discretionary,United States
4,GOOGL|United States,ALPHABET INC CLASS A,2.1885%,Communication,United States
5,AVGO|United States,BROADCOM INC,1.8595%,Information Technology,United States
6,GOOG|United States,ALPHABET INC CLASS C,1.8167%,Communication,United States
7,META|United States,META PLATFORMS INC CLASS A,1.7343%,Communication,United States
8,TSLA|United States,TESLA INC,1.3769%,Consumer Discretionary,United States
9,LLY|United States,ELI LILLY,0.9780%,Health Care,United States


In [5]:
# ACWI benchmark — fetch SSAC + sub-ETFs for look-through

print('Fetching SSAC (ACWI benchmark)...', end=' ')
acwi_raw, acwi_date = fetch_ishares_holdings('SSAC')
acwi_equities = acwi_raw[acwi_raw.apply(is_equity, axis=1)]
print(f'{len(acwi_raw)} total, {len(acwi_equities)} equities (as of {acwi_date})')

all_sub_holdings = {}
for sub_ticker in SSAC_SUB_ETFS:
    print(f'  Sub-ETF {sub_ticker}...', end=' ')
    sub_df, sub_date = fetch_ishares_holdings(sub_ticker)
    all_sub_holdings[sub_ticker] = sub_df
    sub_eq = sub_df[sub_df.apply(is_equity, axis=1)]
    print(f'{len(sub_eq)} equities (as of {sub_date})')
    time.sleep(0.5)

# Build ACWI portfolio with look-through
acwi_eq = acwi_raw[acwi_raw.apply(is_equity, axis=1)].copy()
sub_etf_mask = acwi_eq['Ticker'].isin(SSAC_SUB_ETFS)
acwi_direct = acwi_eq[~sub_etf_mask]
acwi_sub_etfs = acwi_eq[sub_etf_mask]

acwi_stocks = []
for _, h in acwi_direct.iterrows():
    acwi_stocks.append({
        'stock_id': f"{h['Ticker']}|{h['Location']}",
        'ticker': h['Ticker'], 'name': h['Name'],
        'weight': h['Weight (%)'], 'sector': h['Sector'], 'country': h['Location'],
    })

for _, sub in acwi_sub_etfs.iterrows():
    sub_ticker = sub['Ticker']
    sub_weight = sub['Weight (%)'] / 100.0
    sub_df = all_sub_holdings[sub_ticker]
    sub_eq = sub_df[sub_df.apply(is_equity, axis=1)]
    print(f'  Decomposing {sub_ticker}: {sub["Weight (%)"]:.2f}% → {len(sub_eq)} equities')
    for _, h in sub_eq.iterrows():
        acwi_stocks.append({
            'stock_id': f"{h['Ticker']}|{h['Location']}",
            'ticker': h['Ticker'], 'name': h['Name'],
            'weight': sub_weight * h['Weight (%)'], 'sector': h['Sector'], 'country': h['Location'],
        })

acwi_portfolio = (pd.DataFrame(acwi_stocks)
    .groupby('stock_id')
    .agg({'ticker': 'first', 'name': 'first', 'weight': 'sum', 'sector': 'first', 'country': 'first'})
    .reset_index())

total_acwi = acwi_portfolio['weight'].sum()
acwi_portfolio['weight'] = acwi_portfolio['weight'] / total_acwi * 100
acwi_portfolio = acwi_portfolio.sort_values('weight', ascending=False).reset_index(drop=True)

print(f'\nACWI benchmark: {len(acwi_portfolio):,} unique stocks (as of {acwi_date})')
print(f'\nTop 10:')
display(acwi_portfolio.head(10)[['ticker', 'name', 'weight', 'sector', 'country']]
        .style.format({'weight': '{:.4f}%'}))

Fetching SSAC (ACWI benchmark)... 

1769 total, 1728 equities (as of 10/Mar/2026)
  Sub-ETF NDIA... 

165 equities (as of 10/Mar/2026)


  Sub-ETF CNYA... 

409 equities (as of 10/Mar/2026)


...

33 equities (as of 09/Mar/2026)


  Sub-ETF 4BRZ... 

46 equities (as of 09/Mar/2026)


  Decomposing NDIA: 1.50% → 165 equities
  Decomposing 4BRZ: 0.60% → 46 equities
  Decomposing CNYA: 0.44% → 409 equities
  Decomposing IKSA: 0.34% → 33 equities

ACWI benchmark: 2,376 unique stocks (as of 10/Mar/2026)

Top 10:


,ticker,name,weight,sector,country
0,NVDA,NVIDIA CORP,4.7736%,Information Technology,United States
1,AAPL,APPLE INC,4.0888%,Information Technology,United States
2,MSFT,MICROSOFT CORP,3.0615%,Information Technology,United States
3,AMZN,AMAZON COM INC,2.1854%,Consumer Discretionary,United States
4,GOOGL,ALPHABET INC CLASS A,1.9034%,Communication,United States
5,AVGO,BROADCOM INC,1.6315%,Information Technology,United States
6,GOOG,ALPHABET INC CLASS C,1.6013%,Communication,United States
7,2330,TAIWAN SEMICONDUCTOR MANUFACTURING,1.5308%,Information Technology,Taiwan
8,META,META PLATFORMS INC CLASS A,1.5106%,Communication,United States
9,TSLA,TESLA INC,1.1984%,Consumer Discretionary,United States


In [6]:
# Reconcile TKF stock_ids against ACWI before computing active share
# Many stocks have different ticker formats across providers (BRK/B vs BRKB, 
# 0Y3K|Ireland vs EATON|United States, SPOT|United States vs SPOT|Sweden, etc.)

# Merge preferred → common share classes (treat as same stock for active share purposes)
SHARE_CLASS_MERGE = {
    '005935|Korea (South)': '005930|Korea (South)',  # Samsung Electronics pref → common
}
acwi_portfolio['stock_id'] = acwi_portfolio['stock_id'].replace(SHARE_CLASS_MERGE)
acwi_portfolio = (acwi_portfolio
    .groupby('stock_id')
    .agg({'ticker': 'first', 'name': 'first', 'weight': 'sum', 'sector': 'first', 'country': 'first'})
    .reset_index()
)

acwi_sid_set = set(acwi_portfolio['stock_id'])
acwi_by_ticker_norm = {}   # normalized ticker -> stock_id (unique tickers only)
acwi_by_name = {}          # normalized name -> stock_id

acwi_ticker_counts = acwi_portfolio['stock_id'].str.split('|').str[0].value_counts()
unique_acwi_tickers = set(acwi_ticker_counts[acwi_ticker_counts == 1].index)

for _, r in acwi_portfolio.iterrows():
    sid = r['stock_id']
    ticker = sid.split('|')[0]
    ticker_norm = ticker.replace('/', '').replace(' ', '').replace('.', '')
    if ticker in unique_acwi_tickers:
        acwi_by_ticker_norm[ticker_norm] = sid
    norm_name = normalize_em_name(r['name'])
    acwi_by_name[norm_name] = sid

remap_count = 0
remap_weight = 0
for idx, r in tkf_portfolio.iterrows():
    sid = r['stock_id']
    if sid in acwi_sid_set:
        continue
    ticker = sid.split('|')[0] if '|' in sid else ''
    ticker_norm = ticker.replace('/', '').replace(' ', '').replace('.', '')
    norm_name = normalize_em_name(r['name'])
    new_sid = None

    # Try normalized ticker with name verification
    if ticker_norm and ticker_norm in acwi_by_ticker_norm:
        candidate = acwi_by_ticker_norm[ticker_norm]
        cand_name = normalize_em_name(acwi_portfolio[acwi_portfolio['stock_id'] == candidate].iloc[0]['name'])
        if norm_name[:6] == cand_name[:6]:
            new_sid = candidate

    # Try exact normalized name
    if not new_sid and norm_name in acwi_by_name:
        new_sid = acwi_by_name[norm_name]

    # Try prefix name matching (first 15 chars)
    if not new_sid and len(norm_name) >= 10:
        for acwi_name, acwi_sid in acwi_by_name.items():
            if len(acwi_name) >= 10 and norm_name[:15] == acwi_name[:15]:
                new_sid = acwi_sid
                break

    if new_sid:
        tkf_portfolio.loc[idx, 'stock_id'] = new_sid
        remap_count += 1
        remap_weight += r['weight']

# Re-aggregate after remapping
tkf_portfolio = (tkf_portfolio
    .groupby('stock_id')
    .agg({'name': 'first', 'isin': 'first', 'weight': 'sum', 'sector': 'first', 'country': 'first'})
    .reset_index()
)
print(f'Reconciled {remap_count} stock_ids against ACWI ({remap_weight:.2f}% weight remapped)')
print(f'TKF after reconciliation: {len(tkf_portfolio):,} unique stocks')

# Compute active share: ½ × Σ|w_fund − w_benchmark|
tkf_ids = set(tkf_portfolio['stock_id'])
acwi_ids = set(acwi_portfolio['stock_id'])

merged = pd.merge(
    tkf_portfolio[['stock_id', 'name', 'weight', 'sector', 'country']],
    acwi_portfolio[['stock_id', 'weight']].rename(columns={'weight': 'weight_acwi'}),
    on='stock_id', how='outer',
)

acwi_only_mask = merged['name'].isna()
if acwi_only_mask.any():
    acwi_lookup = acwi_portfolio.set_index('stock_id')
    for idx in merged[acwi_only_mask].index:
        sid = merged.loc[idx, 'stock_id']
        if sid in acwi_lookup.index:
            merged.loc[idx, 'name'] = acwi_lookup.loc[sid, 'name']
            merged.loc[idx, 'sector'] = acwi_lookup.loc[sid, 'sector']
            merged.loc[idx, 'country'] = acwi_lookup.loc[sid, 'country']

merged['weight'] = merged['weight'].fillna(0)
merged['weight_acwi'] = merged['weight_acwi'].fillna(0)
merged['in_tkf'] = merged['stock_id'].isin(tkf_ids)
merged['in_acwi'] = merged['stock_id'].isin(acwi_ids)
merged['abs_diff'] = (merged['weight'] - merged['weight_acwi']).abs()
merged['diff'] = merged['weight'] - merged['weight_acwi']

active_share = merged['abs_diff'].sum() / 2
overlap = len(merged[merged['in_tkf'] & merged['in_acwi']])
tkf_only = merged[merged['in_tkf'] & ~merged['in_acwi']]
acwi_only = merged[~merged['in_tkf'] & merged['in_acwi']]

# Decomposition
weight_excluded = acwi_only['weight_acwi'].sum()
weight_added = tkf_only['weight'].sum()
overlapping = merged[merged['in_tkf'] & merged['in_acwi']]
weight_reduced = overlapping.loc[overlapping['diff'] < 0, 'diff'].abs().sum()

display(metric_box('Active Share — TKF vs MSCI ACWI', f'{active_share:.2f}%',
                   f'½ × Σ|w_fund − w_bench| · {len(tkf_portfolio):,} TKF stocks vs {len(acwi_portfolio):,} ACWI stocks · '
                   f'{overlap:,} overlapping · {len(tkf_only):,} TKF-only · {len(acwi_only):,} ACWI-only'))

decomp = pd.DataFrame([
    {'Component': 'Weight excluded (ACWI stocks not in TKF)', 'Weight %': weight_excluded, 'Stocks': len(acwi_only)},
    {'Component': 'Weight added (TKF stocks not in ACWI)', 'Weight %': weight_added, 'Stocks': len(tkf_only)},
    {'Component': 'Weight reduced (overlapping stocks underweight in TKF)', 'Weight %': weight_reduced,
     'Stocks': int((overlapping['diff'] < 0).sum())},
]).set_index('Component')
display(scrollable_table(decomp, title='Active share decomposition', max_height=200,
                         fmt={'Weight %': '{:.2f}'}))

# --- Weight excluded: ACWI stocks not in TKF ---
excluded_df = acwi_only.sort_values('weight_acwi', ascending=False).copy()
excl_out = excluded_df[['name', 'weight_acwi', 'country']].copy()
excl_out.columns = ['Stock', 'ACWI %', 'Country']
excl_out.index = range(1, len(excl_out) + 1)
display(scrollable_table(excl_out, title=f'Weight excluded — {len(acwi_only)} ACWI stocks not in TKF ({weight_excluded:.2f}%)',
                         max_height=350, fmt={'ACWI %': '{:.2f}'}))

# --- Weight added: TKF stocks not in ACWI ---
added_df = tkf_only.sort_values('weight', ascending=False).copy()
add_out = added_df[['name', 'weight', 'country']].copy()
add_out.columns = ['Stock', 'TKF %', 'Country']
add_out.index = range(1, len(add_out) + 1)
display(scrollable_table(add_out, title=f'Weight added — {len(tkf_only)} TKF stocks not in ACWI ({weight_added:.2f}%)',
                         max_height=350, fmt={'TKF %': '{:.2f}'}))

# --- Weight reduced: overlapping stocks underweight in TKF ---
reduced_df = overlapping[overlapping['diff'] < 0].sort_values('diff').copy()
red_out = reduced_df[['name', 'weight', 'weight_acwi', 'diff', 'country']].copy()
red_out.columns = ['Stock', 'TKF %', 'ACWI %', 'Diff', 'Country']
red_out.index = range(1, len(red_out) + 1)
display(scrollable_table(red_out, title=f'Weight reduced — {len(reduced_df)} stocks underweight in TKF ({weight_reduced:.2f}%)',
                         max_height=350, fmt={'TKF %': '{:.2f}', 'ACWI %': '{:.2f}', 'Diff': '{:+.2f}'}))

Reconciled 198 stock_ids against ACWI (4.05% weight remapped)
TKF after reconciliation: 2,656 unique stocks


,Weight %,Stocks
Component,,
Weight excluded (ACWI stocks not in TKF),6.11,752
Weight added (TKF stocks not in ACWI),2.03,1033
Weight reduced (overlapping stocks underweight in TKF),7.87,610


,Stock,ACWI %,Country
1,CHEVRON CORP,0.38,United States
2,PHILIP MORRIS INTERNATIONAL INC,0.28,United States
3,BOEING,0.18,United States
4,HONEYWELL INTERNATIONAL INC,0.16,United States
5,LOCKHEED MARTIN CORP,0.15,United States
6,CONOCOPHILLIPS,0.14,United States
7,BRITISH AMERICAN TOBACCO,0.13,United Kingdom
8,ALTRIA GROUP INC,0.12,United States
9,AIRBUS GROUP,0.12,France
10,SOUTHERN,0.11,United States


,Stock,TKF %,Country
1,AIRBUS,0.10,
2,EMINI S&P500 ESG 03/26 CME,0.03,United States
3,BYD CO LTD-H CNY1,0.02,
4,GRUPO CIBEST SA COP NPV PFD,0.02,
5,BOUBYAN BANK K.S.C KWd100,0.02,
6,Sandisk Corp/DE,0.02,United States
7,EMIRATES NBD PJSC AED1,0.02,
8,KOREA INVESTMENT HOLDINGS CO KRW5000,0.02,
9,PTT PCL-NVDR THB1,0.02,
10,BANCO DE CHILE NPV,0.02,


,Stock,TKF %,ACWI %,Diff,Country
1,TAIWAN SEMICONDUCTOR MANUFAC TWD10,0.58,1.53,-0.95,
2,SAMSUNG ELECTRONICS-PREF KRW5000 PFD,0.08,0.74,-0.65,
3,BERKSHIRE HATHAWAY INC CLASS B,0.39,0.73,-0.33,United States
4,Procter & Gamble Co/The,0.10,0.38,-0.28,United States
5,Coca-Cola Co/The,0.08,0.34,-0.26,United States
6,RTX CORP,0.10,0.29,-0.19,United States
7,PepsiCo Inc,0.06,0.23,-0.17,United States
8,TOYOTA MOTOR CORP,0.05,0.22,-0.17,Japan
9,GE VERNOVA INC,0.13,0.24,-0.11,United States
10,TENCENT HOLDINGS LTD HKD0.00002,0.37,0.48,-0.11,


## Sources of active share

### ESG screening

All ETFs in the TKF model portfolio apply some form of ESG screening. We measure the baseline ESG effect by comparing iShares MSCI ACWI (SSAC, unscreened) with iShares MSCI ACWI ESG Screened — same universe, differing only in exclusions. The excluded stocks' total weight in ACWI gives us the ESG screening baseline.

In [7]:
# ESG screening impact: SSAC (ACWI unscreened) vs iShares MSCI ACWI ESG Screened
# The excluded stocks' total weight = baseline ESG screening effect

ISHARES_PRODUCTS['SACW'] = {'id': 345063, 'slug': '345063'}

print('Fetching iShares MSCI ACWI ESG Screened...', end=' ')
sacw_raw, sacw_date = fetch_ishares_holdings('SACW')
sacw_eq = sacw_raw[sacw_raw.apply(is_equity, axis=1)].copy()
sacw_eq['stock_id'] = sacw_eq.apply(lambda r: f"{r['Ticker']}|{r['Location']}", axis=1)
print(f'{len(sacw_eq)} equities (as of {sacw_date})')

# SSAC (unscreened) — full look-through
ssac_eq = acwi_raw[acwi_raw.apply(is_equity, axis=1)].copy()
ssac_eq = ssac_eq[~ssac_eq['Ticker'].isin(SSAC_SUB_ETFS)]
ssac_eq['stock_id'] = ssac_eq.apply(lambda r: f"{r['Ticker']}|{r['Location']}", axis=1)

ssac_all_stocks = []
for _, h in ssac_eq.iterrows():
    ssac_all_stocks.append({'stock_id': h['stock_id'], 'name': h['Name'], 'weight': h['Weight (%)']})
for sub_ticker in SSAC_SUB_ETFS:
    sub_row = acwi_raw[acwi_raw['Ticker'] == sub_ticker]
    if len(sub_row) == 0:
        continue
    sub_weight_pct = pd.to_numeric(sub_row.iloc[0]['Weight (%)'], errors='coerce') / 100.0
    sub_df = all_sub_holdings[sub_ticker]
    sub_eq = sub_df[sub_df.apply(is_equity, axis=1)]
    for _, h in sub_eq.iterrows():
        ssac_all_stocks.append({
            'stock_id': f"{h['Ticker']}|{h['Location']}",
            'name': h['Name'],
            'weight': sub_weight_pct * h['Weight (%)'],
        })

ssac_full = pd.DataFrame(ssac_all_stocks)
ssac_full = ssac_full.groupby('stock_id').agg({'name': 'first', 'weight': 'sum'}).reset_index()
ssac_total = ssac_full['weight'].sum()
ssac_full['weight_norm'] = ssac_full['weight'] / ssac_total

# Stocks screened out
ssac_ids = set(ssac_full['stock_id'])
sacw_ids = set(sacw_eq['stock_id'])
screened_out = ssac_ids - sacw_ids
screened_out_df = ssac_full[ssac_full['stock_id'].isin(screened_out)].copy()
screened_out_weight = screened_out_df['weight_norm'].sum()

display(metric_box('ESG Screening Baseline',
                   f'{screened_out_weight:.2%} of ACWI weight excluded',
                   f'{len(screened_out)} stocks removed by MSCI ESG screening · '
                   f'ACWI ({len(ssac_full)} stocks) vs ACWI Screened ({len(sacw_eq)} stocks) · '
                   f'TKF active share of 17.37% is ~{17.37 / (screened_out_weight*100):.1f}× this baseline'))

# Table of excluded stocks
excluded = screened_out_df.sort_values('weight_norm', ascending=False).copy()
excluded_display = excluded[['name', 'weight_norm']].copy()
excluded_display.columns = ['Stock', 'ACWI Weight %']
excluded_display.index = range(1, len(excluded_display) + 1)
display(scrollable_table(excluded_display,
                         title='ESG-excluded stocks (in ACWI but not in ACWI Screened)',
                         fmt={'ACWI Weight %': '{:.2%}'}))

Fetching iShares MSCI ACWI ESG Screened... 

1740 equities (as of 09/Mar/2026)


,Stock,ACWI Weight %
1,PROCTER & GAMBLE,0.38%
2,CHEVRON CORP,0.38%
3,COCA-COLA,0.34%
4,RTX CORP,0.29%
5,NESTLE LTD,0.28%
6,PHILIP MORRIS INTERNATIONAL INC,0.28%
7,PEPSICO INC,0.23%
8,BOEING,0.18%
9,HONEYWELL INTERNATIONAL INC,0.16%
10,SIEMENS ENERGY N AG,0.15%


### US equities: Amundi + Xtrackers vs iShares

The TKF model portfolio holds US equities through three ESG-screened ETFs: Xtrackers MSCI USA Screened (17.4%), Amundi MSCI USA Screened (18.5%), and part of iShares MSCI World Screened (SAWD, 19%). All three track variants of MSCI USA ESG Screened — how much additional active share do Amundi and Xtrackers add beyond the iShares baseline?

We compare a 50/50 blend of Amundi + Xtrackers vs the US portion of SAWD. The result isolates provider-level differences (different share classes, slightly different screening dates/criteria) — not the ESG screening effect itself, which was measured above.

In [8]:
# US equities: Amundi + Xtrackers (50/50) vs US portion of iShares SAWD

sawd_us = all_etf_holdings['ishares_SAWD'].copy()
sawd_us = sawd_us[sawd_us['country'] == 'United States']
sawd_us['stock_id'] = sawd_us.apply(lambda r: f"{r['ticker']}|{r['country']}", axis=1)
sawd_us_norm = sawd_us.copy()
sawd_us_total = sawd_us_norm['weight'].sum()
sawd_us_norm['weight'] = sawd_us_norm['weight'] / sawd_us_total

# Build lookups
sawd_ticker_to_sid = {}
sawd_ticker_norm_to_sid = {}
sawd_name_to_sid = {}
for _, r in sawd_us_norm.iterrows():
    sid = r['stock_id']
    sawd_ticker_to_sid[r['ticker']] = sid
    sawd_ticker_norm_to_sid[r['ticker'].replace('/', '').replace(' ', '')] = sid
    sawd_name_to_sid[r['name'].upper().strip()] = sid

xt_usa_h = all_etf_holdings['xtrackers_usa'].copy()
am_usa_h = all_etf_holdings['amundi_usa'].copy()

def map_isin_to_us_sid(isin, name=''):
    if pd.isna(isin) or str(isin) not in isin_to_sid:
        return None
    sid = isin_to_sid[str(isin)]
    ticker = sid.split('|')[0]
    ticker_norm = ticker.replace('/', '').replace(' ', '')
    if ticker in sawd_ticker_to_sid:
        return sawd_ticker_to_sid[ticker]
    if ticker_norm in sawd_ticker_norm_to_sid:
        return sawd_ticker_norm_to_sid[ticker_norm]
    name_key = str(name).upper().strip()
    if name_key in sawd_name_to_sid:
        return sawd_name_to_sid[name_key]
    for sawd_name, sawd_sid in sawd_name_to_sid.items():
        if len(name_key) > 5 and name_key[:20] == sawd_name[:20]:
            return sawd_sid
    return sid

xt_usa_h['stock_id'] = xt_usa_h.apply(lambda r: map_isin_to_us_sid(r['isin'], r.get('name', '')), axis=1)
am_usa_h['stock_id'] = am_usa_h.apply(lambda r: map_isin_to_us_sid(r['isin'], r.get('name', '')), axis=1)
xt_usa_h = xt_usa_h.dropna(subset=['stock_id'])
am_usa_h = am_usa_h.dropna(subset=['stock_id'])

xt_usa_h['weight_blend'] = xt_usa_h['weight'] * 0.5
am_usa_h['weight_blend'] = am_usa_h['weight'] * 0.5

blend = pd.concat([xt_usa_h[['stock_id', 'name', 'weight_blend']], am_usa_h[['stock_id', 'name', 'weight_blend']]])
blend = blend.groupby('stock_id').agg({'name': 'first', 'weight_blend': 'sum'}).reset_index()
blend['weight'] = blend['weight_blend'] / blend['weight_blend'].sum()

merged_us = pd.merge(
    blend[['stock_id', 'name', 'weight']],
    sawd_us_norm[['stock_id', 'weight']].rename(columns={'weight': 'weight_sawd'}),
    on='stock_id', how='outer',
)
merged_us['weight'] = merged_us['weight'].fillna(0)
merged_us['weight_sawd'] = merged_us['weight_sawd'].fillna(0)
merged_us['abs_diff'] = (merged_us['weight'] - merged_us['weight_sawd']).abs()
merged_us['diff'] = merged_us['weight'] - merged_us['weight_sawd']
merged_us['name'] = merged_us['name'].fillna('')
for idx, r in merged_us[merged_us['name'] == ''].iterrows():
    match = sawd_us_norm[sawd_us_norm['stock_id'] == r['stock_id']]
    if len(match):
        merged_us.loc[idx, 'name'] = match.iloc[0]['name']

us_active_share = merged_us['abs_diff'].sum() / 2
overlap_n = len(set(blend['stock_id']) & set(sawd_us_norm['stock_id']))

display(metric_box('US: Amundi + Xtrackers vs iShares SAWD', f'{us_active_share:.2%}',
                   f'{len(blend)} blend stocks vs {len(sawd_us_norm)} SAWD US stocks · {overlap_n} overlapping · '
                   f'Small — same MSCI USA ESG Screened index family'))
display(diff_table(merged_us, 'weight', 'weight_sawd', title='Top weight differences (US)'))

,Stock,Fund %,Bench %,Diff,
1,BERKSHIRE HATHAWAY INC CLASS B,0.63%,1.25%,-0.62%,UW
2,RTX CORP,0.26%,0.00%,+0.26%,OW
3,GE VERNOVA INC,0.21%,0.42%,-0.21%,UW
4,EATON CORP PLC,0.13%,0.00%,+0.13%,OW
5,EATON PLC,0.13%,0.25%,-0.12%,UW
6,ACCENTURE PLC CLASS A,0.12%,0.24%,-0.12%,UW
7,ACCENTURE PLC -A,0.12%,0.00%,+0.12%,OW
8,L3HARRIS TECHNOLOGIES INC,0.07%,0.00%,+0.07%,OW
9,AON PLC-CLASS A,0.07%,0.00%,+0.07%,OW
10,NORFOLK SOUTHERN CORP,0.06%,0.12%,-0.06%,UW


### EM equities: Invesco MSCI EM Universal Screened vs ACWI EM portion

The TKF model portfolio holds EM exposure (10%) through Invesco MSCI EM Universal Screened — which tracks the MSCI EM Universal ESG index, a different variant from the standard MSCI EM index embedded in ACWI. The "Universal" methodology tilts weights toward ESG leaders rather than simply excluding stocks, and applies different screening criteria.

**Data caveat:** Invesco holdings were scraped from companiesmarketcap.com, covering ~93% of the fund's weight. The ~7% missing weight inflates the active share figure — the true value is likely 3–5 pp lower.

In [9]:
# EM equities: Invesco MSCI EM Universal Screened vs EM portion of SSAC (ACWI)

ssac_eq_all = acwi_raw[acwi_raw.apply(is_equity, axis=1)].copy()
ssac_em_direct = ssac_eq_all[ssac_eq_all['Location'].isin(EM_COUNTRIES) & ~ssac_eq_all['Ticker'].isin(SSAC_SUB_ETFS)]

ssac_em_stocks = []
for _, h in ssac_em_direct.iterrows():
    ssac_em_stocks.append({'stock_id': f"{h['Ticker']}|{h['Location']}", 'name': h['Name'],
                           'weight': h['Weight (%)'], 'country': h['Location']})
for sub_ticker in SSAC_SUB_ETFS:
    sub_row = ssac_eq_all[ssac_eq_all['Ticker'] == sub_ticker]
    if len(sub_row) == 0: continue
    sub_weight_pct = sub_row.iloc[0]['Weight (%)'] / 100.0
    sub_df = all_sub_holdings[sub_ticker]
    sub_eq = sub_df[sub_df.apply(is_equity, axis=1)]
    for _, h in sub_eq.iterrows():
        ssac_em_stocks.append({'stock_id': f"{h['Ticker']}|{h['Location']}", 'name': h['Name'],
                               'weight': sub_weight_pct * h['Weight (%)'], 'country': h['Location']})

acwi_em = pd.DataFrame(ssac_em_stocks)
acwi_em = acwi_em.groupby('stock_id').agg({'name': 'first', 'weight': 'sum', 'country': 'first'}).reset_index()
acwi_em_total = acwi_em['weight'].sum()
acwi_em['weight'] = acwi_em['weight'] / acwi_em_total

# Build lookups
acwi_em_sid_set = set(acwi_em['stock_id'])
acwi_em_by_ticker = {}
acwi_em_by_name = {}
acwi_em_by_country_name = {}
for _, r in acwi_em.iterrows():
    sid = r['stock_id']
    acwi_em_by_ticker[sid.split('|')[0]] = sid
    norm_name = normalize_em_name(r['name'])
    acwi_em_by_name[norm_name] = sid
    country = r['country']
    for plen in [15, 10, 6]:
        if len(norm_name) >= plen:
            key = (country, norm_name[:plen])
            if key not in acwi_em_by_country_name:
                acwi_em_by_country_name[key] = sid

ISIN_COUNTRY_MAP = {
    'TW': 'Taiwan', 'KR': 'Korea (South)', 'KY': 'China', 'CN': 'China',
    'IN': 'India', 'ZA': 'South Africa', 'BR': 'Brazil', 'SA': 'Saudi Arabia',
    'MX': 'Mexico', 'AE': 'United Arab Emirates', 'MY': 'Malaysia',
    'TH': 'Thailand', 'ID': 'Indonesia', 'PH': 'Philippines', 'PL': 'Poland',
    'CL': 'Chile', 'QA': 'Qatar', 'KW': 'Kuwait', 'CO': 'Colombia',
    'PE': 'Peru', 'CZ': 'Czech Republic', 'EG': 'Egypt', 'GR': 'Greece',
    'HU': 'Hungary', 'TR': 'Turkey', 'HK': 'China', 'BM': 'China',
}

inv_em = all_etf_holdings['companiesmarketcap'].copy()

def map_invesco_to_acwi_em(isin, name=''):
    if pd.isna(isin): return None
    isin_str = str(isin)
    country = ISIN_COUNTRY_MAP.get(isin_str[:2], '')
    sid = isin_to_sid.get(isin_str)
    if sid:
        if sid in acwi_em_sid_set: return sid
        if sid.split('|')[0] in acwi_em_by_ticker: return acwi_em_by_ticker[sid.split('|')[0]]
    norm_name = normalize_em_name(name)
    if norm_name in acwi_em_by_name: return acwi_em_by_name[norm_name]
    if country:
        for plen in [15, 10, 6]:
            if len(norm_name) >= plen:
                key = (country, norm_name[:plen])
                if key in acwi_em_by_country_name: return acwi_em_by_country_name[key]
    return sid if sid else isin_str

inv_em['stock_id'] = inv_em.apply(lambda r: map_invesco_to_acwi_em(r.get('isin'), r.get('name', '')), axis=1)
inv_em = inv_em.dropna(subset=['stock_id'])
inv_em_agg = inv_em.groupby('stock_id').agg({'name': 'first', 'weight': 'sum'}).reset_index()
inv_em_agg['weight'] = inv_em_agg['weight'] / inv_em_agg['weight'].sum()

merged_em = pd.merge(inv_em_agg[['stock_id', 'name', 'weight']],
                     acwi_em[['stock_id', 'weight']].rename(columns={'weight': 'weight_acwi'}),
                     on='stock_id', how='outer')
merged_em['weight'] = merged_em['weight'].fillna(0)
merged_em['weight_acwi'] = merged_em['weight_acwi'].fillna(0)
merged_em['abs_diff'] = (merged_em['weight'] - merged_em['weight_acwi']).abs()
merged_em['diff'] = merged_em['weight'] - merged_em['weight_acwi']
merged_em['name'] = merged_em['name'].fillna('')
for idx, r in merged_em[merged_em['name'] == ''].iterrows():
    match = acwi_em[acwi_em['stock_id'] == r['stock_id']]
    if len(match): merged_em.loc[idx, 'name'] = match.iloc[0]['name']

em_active_share = merged_em['abs_diff'].sum() / 2
overlap_n = len(set(inv_em_agg['stock_id']) & acwi_em_sid_set)

display(metric_box('EM: Invesco vs ACWI EM', f'{em_active_share:.2%}',
                   f'{len(inv_em_agg)} Invesco stocks vs {len(acwi_em)} ACWI EM stocks · {overlap_n} overlapping · '
                   f'Inflated ~3–5pp by incomplete Invesco data (93% coverage)'))
display(diff_table(merged_em, 'weight', 'weight_acwi', title='Top weight differences (EM)'))

,Stock,Fund %,Bench %,Diff,
1,TAIWAN SEMICONDUCTOR MANUFAC TWD10,6.16%,13.24%,-7.08%,UW
2,SAMSUNG ELECTRONICS-PREF KRW5000 PFD,0.86%,5.58%,-4.72%,UW
3,SK HYNIX INC KRW5000,5.25%,3.14%,+2.11%,OW
4,SAMSUNG ELECTRONICS NON VOTING PRE,0.00%,0.78%,-0.78%,UW
5,CHINA CONSTRUCTION BANK-H CNY1,1.74%,0.96%,+0.78%,OW
6,HDFC BANK LIMITED INR1,1.71%,0.95%,+0.76%,OW
7,PDD HOLDINGS ADS INC,0.00%,0.70%,-0.70%,UW
8,CIA VALE DO RIO DOCE SH,0.00%,0.56%,-0.56%,UW
9,DELTA ELECTRONICS INC TWD10,1.24%,0.70%,+0.54%,OW
10,ANGLOGOLD ASHANTI PLC,0.00%,0.52%,-0.52%,UW


### Europe, Japan, Pacific ex Japan: BNP Paribas Min TE vs iShares SAWD

The TKF model portfolio holds developed ex-North America exposure through three BNP Paribas Minimum Tracking Error ETFs: Europe (12.5%), Japan (3.7%), and Pacific ex Japan (1.9%). Unlike the other TKF funds, BNP Min TE ETFs do **not** apply ESG screening — they track unscreened MSCI regional indices with a minimum tracking error mandate.

We compare against the ex-North America portion of iShares MSCI World ESG Screened (SAWD). The high active share here largely reflects the **ESG screening gap**: BNP holds stocks like Nestle, Shell, and Unilever that SAWD excludes.

**Data caveat:** BNP Japan data covers ~96% of holdings. Some large stocks (e.g. Toyota Motor) may be in the unreported 4%.

In [10]:
# BNP Europe + Japan + Pacific ex Japan vs SAWD ex North America

BNP_WEIGHTS = {'bnp_europe': 12.5, 'bnp_japan': 3.7, 'bnp_pacific': 1.9}
bnp_total_weight = sum(BNP_WEIGHTS.values())

bnp_stocks = []
for source, tkf_w in BNP_WEIGHTS.items():
    df = all_etf_holdings[source].copy()
    rel_w = tkf_w / bnp_total_weight
    for _, h in df.iterrows():
        isin_str = str(h.get('isin', '')) if pd.notna(h.get('isin', '')) else ''
        sid = isin_to_sid.get(isin_str) if isin_str else None
        bnp_stocks.append({'stock_id': sid if sid else isin_str, 'name': h['name'],
                           'isin': isin_str, 'weight': rel_w * h['weight']})

bnp = pd.DataFrame(bnp_stocks)
bnp = bnp.groupby('stock_id').agg({'name': 'first', 'isin': 'first', 'weight': 'sum'}).reset_index()
bnp['weight'] = bnp['weight'] / bnp['weight'].sum()

NA_COUNTRIES = {'United States', 'Canada'}
sawd_exna = all_etf_holdings['ishares_SAWD'].copy()
sawd_exna = sawd_exna[~sawd_exna['country'].isin(NA_COUNTRIES)]
sawd_exna['stock_id'] = sawd_exna.apply(lambda r: f"{r['ticker']}|{r['country']}", axis=1)
sawd_exna_total = sawd_exna['weight'].sum()
sawd_exna['weight'] = sawd_exna['weight'] / sawd_exna_total

sawd_exna_sid_set = set(sawd_exna['stock_id'])
sawd_exna_by_ticker = {}
sawd_exna_by_ticker_norm = {}
sawd_exna_by_name = {}
sawd_exna_by_country_name = {}
for _, r in sawd_exna.iterrows():
    sid, ticker, country = r['stock_id'], r['ticker'], r['country']
    sawd_exna_by_ticker[ticker] = sid
    sawd_exna_by_ticker_norm[ticker.replace('/', '').replace(' ', '')] = sid
    norm_name = normalize_em_name(r['name'])
    sawd_exna_by_name[norm_name] = sid
    for plen in [15, 10]:
        if len(norm_name) >= plen:
            key = (country, norm_name[:plen])
            if key not in sawd_exna_by_country_name:
                sawd_exna_by_country_name[key] = sid

ISIN_DM_COUNTRY_MAP = {
    'GB': 'United Kingdom', 'DE': 'Germany', 'FR': 'France', 'CH': 'Switzerland',
    'NL': 'Netherlands', 'SE': 'Sweden', 'DK': 'Denmark', 'IT': 'Italy',
    'ES': 'Spain', 'FI': 'Finland', 'NO': 'Norway', 'BE': 'Belgium',
    'IE': 'Ireland', 'AT': 'Austria', 'PT': 'Portugal', 'LU': 'Luxembourg',
    'JP': 'Japan', 'AU': 'Australia', 'HK': 'Hong Kong', 'SG': 'Singapore',
    'NZ': 'New Zealand', 'IL': 'Israel',
}

def map_bnp_to_sawd(stock_id, isin, name):
    ticker = stock_id.split('|')[0] if '|' in stock_id else ''
    ticker_norm = ticker.replace('/', '').replace(' ', '')
    country = ISIN_DM_COUNTRY_MAP.get(isin[:2], '') if len(isin) >= 2 else ''
    if stock_id in sawd_exna_sid_set: return stock_id
    if ticker in sawd_exna_by_ticker: return sawd_exna_by_ticker[ticker]
    if ticker_norm in sawd_exna_by_ticker_norm: return sawd_exna_by_ticker_norm[ticker_norm]
    norm_name = normalize_em_name(name)
    if norm_name in sawd_exna_by_name: return sawd_exna_by_name[norm_name]
    if country:
        for plen in [15, 10]:
            if len(norm_name) >= plen:
                key = (country, norm_name[:plen])
                if key in sawd_exna_by_country_name: return sawd_exna_by_country_name[key]
    return stock_id

bnp['stock_id_matched'] = bnp.apply(lambda r: map_bnp_to_sawd(r['stock_id'], r['isin'], r['name']), axis=1)
bnp_agg = bnp.groupby('stock_id_matched').agg({'name': 'first', 'weight': 'sum'}).reset_index()
bnp_agg = bnp_agg.rename(columns={'stock_id_matched': 'stock_id'})
bnp_agg['weight'] = bnp_agg['weight'] / bnp_agg['weight'].sum()

merged_bnp = pd.merge(bnp_agg[['stock_id', 'name', 'weight']],
                      sawd_exna[['stock_id', 'weight']].rename(columns={'weight': 'weight_sawd'}),
                      on='stock_id', how='outer')
merged_bnp['weight'] = merged_bnp['weight'].fillna(0)
merged_bnp['weight_sawd'] = merged_bnp['weight_sawd'].fillna(0)
merged_bnp['abs_diff'] = (merged_bnp['weight'] - merged_bnp['weight_sawd']).abs()
merged_bnp['diff'] = merged_bnp['weight'] - merged_bnp['weight_sawd']
merged_bnp['name'] = merged_bnp['name'].fillna('')
for idx, r in merged_bnp[merged_bnp['name'] == ''].iterrows():
    match = sawd_exna[sawd_exna['stock_id'] == r['stock_id']]
    if len(match): merged_bnp.loc[idx, 'name'] = match.iloc[0]['name']

bnp_active_share = merged_bnp['abs_diff'].sum() / 2
overlap_n = len(set(bnp_agg['stock_id']) & sawd_exna_sid_set)
bnp_only_n = len(set(bnp_agg['stock_id']) - sawd_exna_sid_set)

display(metric_box('DM ex-NA: BNP Min TE vs SAWD ex-NA', f'{bnp_active_share:.2%}',
                   f'{len(bnp_agg)} BNP stocks vs {len(sawd_exna)} SAWD ex-NA stocks · {overlap_n} overlapping · '
                   f'{bnp_only_n} BNP-only (mostly ESG exclusions held by unscreened BNP funds) · '
                   f'BNP Japan ~96% coverage'))
display(diff_table(merged_bnp, 'weight', 'weight_sawd', title='Top weight differences (DM ex-NA)'))

,Stock,Fund %,Bench %,Diff,
1,NESTLE SA,1.36%,0.00%,+1.36%,OW
2,TOYOTA MOTOR CORP,0.00%,1.11%,-1.11%,UW
3,SHELL PLC,1.08%,0.00%,+1.08%,OW
4,UNILEVER PLC,0.76%,0.00%,+0.76%,OW
5,SIEMENS ENERGY N AG,0.66%,0.00%,+0.66%,OW
6,AIRBUS,0.57%,0.00%,+0.57%,OW
7,SAFRAN SA,0.54%,0.00%,+0.54%,OW
8,COMPAGNIE FINANCIERE RICHEMONT SA,0.50%,0.00%,+0.50%,OW
9,BP PLC,0.40%,0.00%,+0.40%,OW
10,INDUSTRIA DE DISENO TEXTIL SA,0.35%,0.00%,+0.35%,OW


### Different index: Vanguard ESG North America All Cap

Vanguard ESG North America All Cap (15.4% of TKF) tracks the FTSE North America All Cap Choice Index — a different index family from the MSCI-based ETFs used by the other 8 funds. This creates active share through two channels:

1. **Small-cap stocks** — Vanguard's all-cap mandate includes ~1,000 small caps not in MSCI ACWI (large+mid cap only)
2. **Different ESG screening** — FTSE ESG excludes different companies than MSCI ESG Screened

We compare Vanguard vs Xtrackers USA + Canada (both held in TKF) to isolate what the Vanguard allocation adds beyond the MSCI-based funds.

**Data caveat:** Vanguard holdings are from 31 Jan 2026 (stale by ~6 weeks). The weight differences on overlapping stocks are partly due to price movements since then.

In [11]:
# Vanguard ESG NA All Cap vs Xtrackers USA + Canada

acwi_us_weight = acwi_portfolio[acwi_portfolio['country'] == 'United States']['weight'].sum()
acwi_ca_weight = acwi_portfolio[acwi_portfolio['country'] == 'Canada']['weight'].sum()
us_share = acwi_us_weight / (acwi_us_weight + acwi_ca_weight)
ca_share = 1 - us_share

xt_usa = all_etf_holdings['xtrackers_usa'].copy()
xt_can = all_etf_holdings['xtrackers_canada'].copy()
xt_usa['weight_combined'] = xt_usa['weight'] * us_share
xt_can['weight_combined'] = xt_can['weight'] * ca_share

def get_sid(row):
    isin = row.get('isin', '')
    if pd.notna(isin) and str(isin) in isin_to_sid:
        return isin_to_sid[str(isin)]
    return str(isin)

xt_combined = pd.concat([
    xt_usa[['name', 'isin', 'weight_combined']].assign(stock_id=xt_usa.apply(get_sid, axis=1)),
    xt_can[['name', 'isin', 'weight_combined']].assign(stock_id=xt_can.apply(get_sid, axis=1)),
])
xt_combined = xt_combined.groupby('stock_id').agg({'name': 'first', 'weight_combined': 'sum'}).reset_index()
xt_combined['weight'] = xt_combined['weight_combined'] / xt_combined['weight_combined'].sum()

vg = all_etf_holdings['vanguard_na'].copy()
vg['stock_id'] = vg.apply(lambda r: f"{r['ticker']}|{r['country']}" if pd.notna(r.get('ticker')) and pd.notna(r.get('country')) else r['name'], axis=1)
vg_portfolio = vg.groupby('stock_id').agg({'name': 'first', 'weight': 'sum'}).reset_index()
vg_portfolio['weight'] = vg_portfolio['weight'] / vg_portfolio['weight'].sum()

xt_ids = set(xt_combined['stock_id'])
vg_ids = set(vg_portfolio['stock_id'])
vg_only = vg_ids - xt_ids
vg_only_weight = vg_portfolio[vg_portfolio['stock_id'].isin(vg_only)]['weight'].sum()

merged_vx = pd.merge(
    vg_portfolio[vg_portfolio['stock_id'].isin(xt_ids & vg_ids)][['stock_id', 'name', 'weight']],
    xt_combined[xt_combined['stock_id'].isin(xt_ids & vg_ids)][['stock_id', 'weight']].rename(columns={'weight': 'weight_xt'}),
    on='stock_id',
)
merged_vx['abs_diff'] = (merged_vx['weight'] - merged_vx['weight_xt']).abs()
merged_vx['diff'] = merged_vx['weight'] - merged_vx['weight_xt']
stale_active_share = merged_vx['abs_diff'].sum() / 2

display(metric_box('Vanguard ESG NA All Cap vs Xtrackers USA+Canada', '',
                   f'{len(vg_portfolio):,} Vanguard stocks vs {len(xt_combined)} Xtrackers · '
                   f'{len(xt_ids & vg_ids)} overlapping · '
                   f'{len(vg_only):,} Vanguard-only ({vg_only_weight:.2%} — mostly small caps) · '
                   f'Overlapping stocks weight diff (stale prices): {stale_active_share:.2%}'))
display(diff_table(merged_vx, 'weight', 'weight_xt', title='Top weight differences on overlapping stocks (Vanguard vs Xtrackers)'))

,Stock,Fund %,Bench %,Diff,
1,Microsoft Corp,5.75%,5.04%,+0.71%,OW
2,Amazon.com Inc,4.14%,3.58%,+0.55%,OW
3,Alphabet Inc,3.54%,3.11%,+0.43%,OW
4,Meta Platforms Inc,2.82%,2.46%,+0.36%,OW
5,NVIDIA Corp,8.06%,7.74%,+0.32%,OW
6,Alphabet Inc,2.89%,2.60%,+0.29%,OW
7,Tesla Inc,2.19%,1.97%,+0.22%,OW
8,Apple Inc,6.82%,6.66%,+0.15%,OW
9,Advanced Micro Devices Inc,0.69%,0.58%,+0.11%,OW
10,Netflix Inc,0.64%,0.73%,-0.09%,UW
